# MinerU — PDF → Markdown / HTML (local, Apple Silicon)

Experiment with [**MinerU**](https://github.com/opendatalab/mineru) (opendatalab), a one-stop
open-source document-extraction tool that turns **PDF / image / DOCX / PPTX / XLSX** into
**Markdown + JSON**, preserving reading order and extracting text, tables (as HTML), formulas
(LaTeX), and figures.

Unlike the PaddleOCR / OvisOCR2 notebooks in this repo (which need a Kaggle GPU), **MinerU's
`pipeline` backend runs on CPU and Apple-Silicon (MPS)** — so this notebook is meant to run
**locally on your Mac**. No Kaggle round-trip.

**What this notebook does**, for every PDF in `../../pdf_resources/`:

| artifact | what it is |
|---|---|
| `out/<stem>/.../<stem>.md` | the Markdown conversion (tables as HTML, formulas as LaTeX) |
| `out/<stem>/.../<stem>_content_list.json` | flat, layout-free list of content blocks (text/table/image/equation…) |
| `out/<stem>/.../images/` | extracted figures referenced by the Markdown |
| `out/<stem>.html` | the Markdown rendered to a standalone HTML page (MathJax for formulas) |
| `mineru_manifest.csv` | one row per PDF: backend, latency, pages, chars, #tables, #images, #equations |

MinerU version targeted: **v3.4** (2026-06). Tested path: `pipeline` backend on Apple-Silicon.

## 0. Environment (read first — one-time setup)

MinerU needs **Python 3.10–3.13**. Your system `python3` is 3.9, so this notebook must run
inside a dedicated venv. From the repo root, in a terminal:

```bash
# 1. a modern Python (Homebrew)
brew install python@3.12

# 2. a venv just for MinerU
/opt/homebrew/bin/python3.12 -m venv .venv-mineru
source .venv-mineru/bin/activate

# 3. MinerU (core = pipeline backend, which is all we need on Apple Silicon)
python -m pip install --upgrade pip
pip install -U "mineru[core]"

# 4. make this venv a Jupyter kernel, then launch
pip install jupyter ipykernel markdown
python -m ipykernel install --user --name mineru --display-name "Python (mineru)"
jupyter lab   # or: jupyter notebook
```

Then open this notebook and pick the **Python (mineru)** kernel (top-right kernel picker).


The **first conversion downloads the pipeline models** (~1–2 GB: layout, OCR, table, formula)
into `~/.cache`. That happens once; later runs are offline. On a 16 GB M-series Mac the
`pipeline` backend is the safe default — the `vlm` / `hybrid` backends want ~8 GB of *free*
unified memory and are heavier.

## 1. Sanity check — right Python, MinerU importable, device

In [1]:
import sys, platform, shutil, subprocess

print('Python :', sys.version.split()[0], '(need 3.10–3.13)')
assert sys.version_info[:2] >= (3, 10), (
    'Wrong kernel. Select the "Python (mineru)" kernel — see section 0.'
)

# MinerU is driven via its CLI (stable across versions); confirm it's on PATH.
mineru_bin = shutil.which('mineru')
assert mineru_bin, 'mineru CLI not found — did you `pip install "mineru[core]"` in this venv?'
print('mineru :', mineru_bin)
try:
    import mineru
    print('version:', getattr(mineru, '__version__', '?'))
except Exception as e:
    print('(import mineru failed, CLI still usable):', e)

# Device: Apple-Silicon → mps, NVIDIA → cuda, else cpu.
def detect_device():
    try:
        import torch
        if torch.cuda.is_available():
            return 'cuda'
        if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
            return 'mps'
    except Exception:
        pass
    return 'cpu'

print('machine:', platform.platform())
print('device :', detect_device())

Python : 3.12.13 (need 3.10–3.13)
mineru : /Users/asikifthakerhamim/Documents/ocr_exploration/.venv-mineru/bin/mineru
version: ?
machine: macOS-26.5.2-arm64-arm-64bit
device : mps


## 2. Config

Everything is relative to this notebook's folder (`notebooks/mineru/`), so the PDFs resolve to
`ocr_exploration/pdf_resources/`. Tune the knobs below.

In [2]:
from pathlib import Path

HERE      = Path.cwd()
# Robust to being launched from repo root or from notebooks/mineru/.
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'pdf_resources').is_dir()), HERE)
PDF_DIR   = REPO_ROOT / 'pdf_resources'
OUT_DIR   = HERE / 'out'
HTML_DIR  = HERE / 'html'

BACKEND   = 'pipeline'   # 'pipeline' (CPU/MPS, safe) | 'vlm' | 'hybrid' (need ~8GB free)
DEVICE    = detect_device()   # override e.g. 'cpu' if MPS misbehaves
LANG      = 'en'         # OCR language hint for pipeline backend

# The Atlanta Code of Ordinances is hundreds of pages — cap pages for a fast first pass.
# Set to None to convert every page of every PDF.
MAX_PAGES = 15           # e.g. 15 → only pages 0..14 per PDF; None → all pages
MAX_FILES = None         # e.g. 2 → smoke-test the first 2 PDFs; None → all

OUT_DIR.mkdir(exist_ok=True)
HTML_DIR.mkdir(exist_ok=True)
print('PDF_DIR:', PDF_DIR)
print('OUT_DIR:', OUT_DIR)
print('backend/device/lang:', BACKEND, DEVICE, LANG)

PDF_DIR: /Users/asikifthakerhamim/Documents/ocr_exploration/pdf_resources
OUT_DIR: /Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/mineru/out
backend/device/lang: pipeline mps en


## 3. Discover the PDFs

In [3]:
pdfs = sorted(PDF_DIR.glob('*.pdf'))
if MAX_FILES:
    pdfs = pdfs[:MAX_FILES]
assert pdfs, f'No PDFs under {PDF_DIR}'

for p in pdfs:
    print(f'{p.stat().st_size/1024:8.0f} KB  {p.name}')
print(f'\n{len(pdfs)} PDF(s)')

     787 KB  AF SE 2.pdf
     206 KB  Admin Eval Framework.docx.pdf
     666 KB  Agenda_ Operational Leadership Institute (SY 26-27).pdf
     274 KB  Athlos 2024-2025 Observation Rubric Teacher.docx.pdf
    2076 KB  Atlanta, GA Code of Ordinances.pdf
     224 KB  Benefits_HR Specialist Evaluation Form Template  (1).pdf
     224 KB  Benefits_HR Specialist Evaluation Form Template .pdf
      81 KB  Bullseye Math Checklist  - Sheet1.pdf
    2431 KB  Classroom Environment Checklist 6-8.docx.pdf
     130 KB  Marzano Rubrics with scales. potential evidence, and elements 2022.pdf

10 PDF(s)


## 4. Convert each PDF with MinerU

We call the **`mineru` CLI** (its API is stable across releases, unlike the internal Python
functions). Per file we time the run and write everything under `out/<stem>/`.

```
mineru -p <pdf> -o out/<stem> -b pipeline -l en -d mps [-e 14]
```

Flags: `-p` input, `-o` output dir, `-b` backend, `-l` OCR language, `-d` device,
`-e` end page (0-indexed, inclusive) when `MAX_PAGES` is set.

The first file also triggers the one-time model download, so it will be slow; watch the
streamed CLI log below it.

In [ ]:
import time, shutil

def safe_name(stem: str) -> str:
    # shared converter standard: sanitize the per-PDF folder name
    for a, b in [('(', '-'), (')', '-'), ('[', '-'), (']', '-'), (' ', '_')]:
        stem = stem.replace(a, b)
    for d in '‐‑‒–—―−':
        stem = stem.replace(d, '-')
    return stem

def convert(pdf: Path) -> dict:
    stem = pdf.stem
    safe = safe_name(stem)
    out  = OUT_DIR / safe          # standard: out/<safe_stem>/
    raw  = out / 'raw'
    if out.exists():
        shutil.rmtree(out)
    raw.mkdir(parents=True, exist_ok=True)

    # MinerU writes its native nested tree (<stem>/auto/...) under raw/.
    cmd = ['mineru', '-p', str(pdf), '-o', str(raw),
           '-b', BACKEND, '-l', LANG, '-d', DEVICE]
    if MAX_PAGES is not None:
        cmd += ['-e', str(MAX_PAGES - 1)]   # -e is 0-indexed inclusive
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.perf_counter() - t0
    ok = proc.returncode == 0

    md_path = cl_path = ''
    if ok:
        # promote the primary artifacts up to the standard locations
        md = next(iter(sorted(raw.rglob('*.md'))), None)
        if md:
            (out / f'{safe}.md').write_text(md.read_text(encoding='utf-8'), encoding='utf-8')
            md_path = str(out / f'{safe}.md')
            imgs = md.parent / 'images'
            if imgs.is_dir():                       # move so images aren't duplicated in raw/
                shutil.move(str(imgs), str(out / 'images'))
        cl = next(iter(sorted(raw.rglob('*_content_list.json'))), None)
        if cl:
            (out / f'{safe}.content_list.json').write_text(cl.read_text(encoding='utf-8'), encoding='utf-8')
            cl_path = str(out / f'{safe}.content_list.json')
    else:
        print(f'  !! {stem} failed (rc={proc.returncode})')
        print('  ' + (proc.stderr or proc.stdout or '')[-800:].replace('\n', '\n  '))

    return {'stem': stem, 'safe': safe, 'ok': ok, 'latency_s': round(dt, 2),
            'md_path': md_path, 'cl_path': cl_path,
            'stdout': proc.stdout, 'stderr': proc.stderr}

runs = []
for i, pdf in enumerate(pdfs, 1):
    print(f'[{i}/{len(pdfs)}] {pdf.name} …', flush=True)
    r = convert(pdf)
    if r['ok']:
        print(f'      done in {r["latency_s"]}s')
    runs.append(r)
print('\nconversion pass complete')

## 5. Collect outputs & compute per-PDF stats

`convert()` already promoted each run into the **shared converter standard** —
`out/<safe_stem>/<safe_stem>.md`, `<safe_stem>.content_list.json`, `images/`, and the native
MinerU extras under `raw/`. Here we just tally content-block types from the content list.

In [ ]:
import json
from collections import Counter

def block_stats(content_list_path):
    if not content_list_path:
        return Counter(), 0
    try:
        blocks = json.loads(Path(content_list_path).read_text(encoding='utf-8'))
    except Exception:
        return Counter(), 0
    kinds = Counter(b.get('type', '?') for b in blocks if isinstance(b, dict))
    pages = 1 + max((b.get('page_idx', 0) for b in blocks if isinstance(b, dict)), default=-1)
    return kinds, pages

# Standard layout: out/<safe_stem>/<safe_stem>.md + <safe_stem>.content_list.json
# (paths were promoted by convert(); native MinerU extras live under out/<safe_stem>/raw/)
records = []
for r in runs:
    md_path = r.get('md_path', '')
    cl_path = r.get('cl_path', '')
    text = Path(md_path).read_text(encoding='utf-8') if md_path else ''
    kinds, pages = block_stats(cl_path)
    records.append({
        'pdf': r['stem'],
        'ok': r['ok'],
        'backend': BACKEND,
        'latency_s': r['latency_s'],
        'pages': pages,
        'chars': len(text),
        'n_tables': kinds.get('table', 0),
        'n_images': kinds.get('image', 0),
        'n_equations': kinds.get('equation', 0),
        'n_text': kinds.get('text', 0),
        'md_path': md_path,
    })

try:
    import pandas as pd
    df = pd.DataFrame(records)
    display(df.drop(columns=['md_path']))
except Exception:
    df = None
    for rec in records:
        print(rec)

## 6. Markdown → standalone HTML

MinerU emits Markdown with raw HTML tables + LaTeX. We render it to a full HTML page — the
**same house CSS** used by the other notebooks in this repo — with MathJax so formulas display,
and rewrite image paths so extracted figures load.

In [6]:
import markdown as md_lib

HOUSE_CSS = (
    'body{font-family:Arial,Helvetica,sans-serif;max-width:900px;margin:24px auto;'
    'padding:0 16px;line-height:1.5}'
    'table{border-collapse:collapse;margin:12px 0}'
    'td,th{border:1px solid #888;padding:4px 8px;vertical-align:top}'
    'h1,h2,h3{margin:.6em 0 .3em}img{max-width:100%}'
)
MATHJAX = (
    "<script>window.MathJax={tex:{inlineMath:[['$','$'],['\\\\(','\\\\)']],"
    "displayMath:[['$$','$$'],['\\\\[','\\\\]']]}};</script>"
    '<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>'
)

def to_html(md_path: Path, stem: str) -> Path:
    raw = md_path.read_text(encoding='utf-8')
    body = md_lib.markdown(raw, extensions=['tables', 'fenced_code'])
    # images in the .md are relative to the .md's folder; make them absolute file URLs
    base = md_path.parent.resolve().as_uri()
    body = body.replace('src="images/', f'src="{base}/images/')
    html = (f'<!doctype html><html><head><meta charset="utf-8">'
            f'<title>{stem}</title><style>{HOUSE_CSS}</style>{MATHJAX}'
            f'</head><body>{body}</body></html>')
    out = HTML_DIR / f'{stem}.html'
    out.write_text(html, encoding='utf-8')
    return out

for rec in records:
    if rec['md_path']:
        to_html(Path(rec['md_path']), rec['pdf'])
print('wrote', len(list(HTML_DIR.glob('*.html'))), 'HTML pages to', HTML_DIR)

wrote 10 HTML pages to /Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/mineru/html


## 7. Preview a result inline

Change `PREVIEW` to any PDF stem from the table above.

In [8]:
from IPython.display import HTML, display

PREVIEW = records[0]['pdf'] if records else None
page = HTML_DIR / f'{PREVIEW}.html'
if PREVIEW and page.exists():
    print(page)
    display(HTML(page.read_text(encoding='utf-8')))
else:
    print('nothing to preview')

/Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/mineru/html/AF SE 2.html
